In [7]:
# Cell 1: Install dependencies (uncomment to run once)
# %pip install -q numpy pandas scikit-learn openpyxl matplotlib

In [8]:
# Cell 2: Load data
# load-data.py uses a hyphen — importlib handles this gracefully
import importlib.util

_spec = importlib.util.spec_from_file_location(
    'load_data',
    r'C:\Users\Lenovo\Desktop\anfis-bn-code\load-data.py'
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

DATA_PATH = r'C:\Users\Lenovo\Desktop\anfis-bn-code\last_30s_segments'
X_train, y_train, user_train, X_test, y_test, user_test = _mod.load_data(DATA_PATH)

X_train:(632, 4, 30)  X_test:(193, 4, 30)
y_train  rest=317  task=315
y_test   rest=95   task=98


In [9]:
# Cell 3: Imports & pipeline configuration
import importlib
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score

import fts_encoder, anfis_tsk, bn_fusion, eval_utils
importlib.reload(fts_encoder)
importlib.reload(anfis_tsk)
importlib.reload(bn_fusion)
importlib.reload(eval_utils)

from fts_encoder import FTSEncoder
from anfis_tsk   import ANFIS_TSK
from bn_fusion   import BNFusion, SIGNAL_NAMES, BIN_NAMES
from eval_utils  import print_metrics, find_best_threshold, print_config

# ---------- Hyper-parameters (single place to tweak) ----------
N_SETS         = 3            # fuzzy sets per signal  (Low / Med / High)
MF_TYPE        = "triangular"      # "triangular" | "gaussian" | "trapezoid" | "gbell"
MF_SIGMA_SCALE = 0.5          # gaussian σ = spacing * this;  gbell a = spacing * this
GBELL_B        = 2.0          # gbell shape exponent (only used when MF_TYPE="gbell")
N_RULES        = 8            # ANFIS rules per signal (KMeans clusters)
SIGMA_SCALE    = 1.5          # ANFIS Gaussian width multiplier
RIDGE_LAMBDA   = 0.1          # Ridge regularisation for WLS
N_EPOCHS       = 15           # Hybrid-learning iterations (WLS + GD)
LR             = 0.001        # Learning rate for premise GD
N_BINS         = 3            # BN discretisation bins  (low / med / high)
ALPHA          = 1.0          # Laplace smoothing
K_FOLDS        = 5            # GroupKFold splits

CONFIG = {
    "K (CV folds)":             K_FOLDS,
    "MF type (FTS)":            MF_TYPE,
    "MF sigma_scale":           MF_SIGMA_SCALE,
    "gbell_b":                  GBELL_B,
    "n_fuzzy_sets":             N_SETS,
    "n_rules (ANFIS)":          N_RULES,
    "sigma_scale (ANFIS)":      SIGMA_SCALE,
    "ridge_lambda":             RIDGE_LAMBDA,
    "n_hybrid_epochs":          N_EPOCHS,
    "learning_rate (GD)":       LR,
    "normalisation (FTS)":      "total_sum",
    "n_bins (BN)":              N_BINS,
    "binning":                  "quantile (1/3, 2/3)",
    "smoothing_alpha":          ALPHA,
    "threshold_strategy":       "tune_F1_on_OOF",
}
print_config(CONFIG)


  PIPELINE CONFIGURATION
  K (CV folds)                       : 5
  MF type (FTS)                      : triangular
  MF sigma_scale                     : 0.5
  gbell_b                            : 2.0
  n_fuzzy_sets                       : 3
  n_rules (ANFIS)                    : 8
  sigma_scale (ANFIS)                : 1.5
  ridge_lambda                       : 0.1
  n_hybrid_epochs                    : 15
  learning_rate (GD)                 : 0.001
  normalisation (FTS)                : total_sum
  n_bins (BN)                        : 3
  binning                            : quantile (1/3, 2/3)
  smoothing_alpha                    : 1.0
  threshold_strategy                 : tune_F1_on_OOF



In [10]:
# Cell 4: GroupKFold Cross-Validation  (subject-independent)
gkf = GroupKFold(n_splits=K_FOLDS)
oof_probs = np.full(len(y_train), np.nan)

for fold, (tr_idx, va_idx) in enumerate(
    gkf.split(X_train, y_train, user_train)
):
    X_tr, X_va = X_train[tr_idx], X_train[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]

    print(f"\n{'=' * 58}")
    print(f"  FOLD {fold + 1}/{K_FOLDS}")
    print(f"{'=' * 58}")
    print(f"  Train : {len(y_tr):4d}  (rest={int((y_tr==0).sum())}, task={int((y_tr==1).sum())})")
    print(f"  Val   : {len(y_va):4d}  (rest={int((y_va==0).sum())}, task={int((y_va==1).sum())})")

    # ---- Step 1: FTS Encoding ----
    fts = FTSEncoder(n_sets=N_SETS, mf_type=MF_TYPE,
                     mf_sigma_scale=MF_SIGMA_SCALE, gbell_b=GBELL_B)
    F_tr = fts.fit_transform(X_tr)
    F_va = fts.transform(X_va)
    print(f"  FTS features  : train {F_tr.shape}  val {F_va.shape}")

    zero_pct_tr = (F_tr == 0).sum() / F_tr.size * 100
    zero_pct_va = (F_va == 0).sum() / F_va.size * 100
    print(f"  FTS sparsity  : train {zero_pct_tr:.1f}% zeros, "
          f"val {zero_pct_va:.1f}% zeros")

    # ---- Step 2: 4 × ANFIS (one per signal) ----
    ap_tr = np.zeros((len(y_tr), 4))
    ap_va = np.zeros((len(y_va), 4))

    for s in range(4):
        mdl = ANFIS_TSK(
            n_rules=N_RULES, sigma_scale=SIGMA_SCALE,
            ridge_lambda=RIDGE_LAMBDA, n_epochs=N_EPOCHS,
            lr=LR, random_state=42,
        )
        mdl.fit(F_tr[:, s * 9:(s + 1) * 9], y_tr)
        ap_tr[:, s] = mdl.predict_proba(F_tr[:, s * 9:(s + 1) * 9])
        ap_va[:, s] = mdl.predict_proba(F_va[:, s * 9:(s + 1) * 9])
        print(f"    ANFIS [{SIGNAL_NAMES[s]:4s}]  val proba "
              f"[{ap_va[:, s].min():.3f} .. {ap_va[:, s].max():.3f}]")

    print(f"  ANFIS outputs : train {ap_tr.shape}  val {ap_va.shape}")

    # ---- Step 3: BN Fusion ----
    bn = BNFusion(n_bins=N_BINS, alpha=ALPHA)
    bn.fit(ap_tr, y_tr)
    ev_va = bn.get_evidence_bins(ap_va)
    print(f"  BN evidence   : val {ev_va.shape}")

    va_post = bn.predict_proba(ap_va)
    oof_probs[va_idx] = va_post

    va_pred = (va_post >= 0.5).astype(int)
    print(f"  Fold Acc={accuracy_score(y_va, va_pred):.4f}  "
          f"F1={f1_score(y_va, va_pred):.4f}")

print("\n  All folds completed.")


  FOLD 1/5
  Train :  504  (rest=251, task=253)
  Val   :  128  (rest=66, task=62)
  FTS features  : train (504, 36)  val (128, 36)
  FTS sparsity  : train 48.3% zeros, val 50.1% zeros
    ANFIS [Temp]  val proba [0.589 .. 0.632]
    ANFIS [HR  ]  val proba [0.571 .. 0.638]
    ANFIS [GSR ]  val proba [0.542 .. 0.644]
    ANFIS [RR  ]  val proba [0.501 .. 0.713]
  ANFIS outputs : train (504, 4)  val (128, 4)
  BN evidence   : val (128, 4)
  Fold Acc=0.5391  F1=0.5203

  FOLD 2/5
  Train :  517  (rest=260, task=257)
  Val   :  115  (rest=57, task=58)
  FTS features  : train (517, 36)  val (115, 36)
  FTS sparsity  : train 46.7% zeros, val 60.9% zeros
    ANFIS [Temp]  val proba [0.597 .. 0.638]
    ANFIS [HR  ]  val proba [0.493 .. 0.633]
    ANFIS [GSR ]  val proba [0.589 .. 0.631]
    ANFIS [RR  ]  val proba [0.382 .. 0.705]
  ANFIS outputs : train (517, 4)  val (115, 4)
  BN evidence   : val (115, 4)
  Fold Acc=0.6261  F1=0.6950

  FOLD 3/5
  Train :  518  (rest=260, task=258)
  Val

In [11]:
# Cell 5: OOF evaluation (default + tuned threshold)
oof_pred_default = (oof_probs >= 0.5).astype(int)
print_metrics(y_train, oof_pred_default,
              title="OOF Results  (threshold = 0.50)")

best_thr, best_f1 = find_best_threshold(y_train, oof_probs)
print(f"Best OOF threshold : {best_thr:.2f}  (F1 = {best_f1:.4f})")

oof_pred_tuned = (oof_probs >= best_thr).astype(int)
print_metrics(y_train, oof_pred_tuned,
              title=f"OOF Results  (threshold = {best_thr:.2f})")


  OOF Results  (threshold = 0.50)
  Accuracy : 0.5744
  F1-score : 0.5710

  Confusion Matrix:
                 Predicted
                 rest  task
  Actual rest   184   133
  Actual task   136   179

              precision    recall  f1-score   support

        rest       0.57      0.58      0.58       317
        task       0.57      0.57      0.57       315

    accuracy                           0.57       632
   macro avg       0.57      0.57      0.57       632
weighted avg       0.57      0.57      0.57       632

Best OOF threshold : 0.23  (F1 = 0.6727)

  OOF Results  (threshold = 0.23)
  Accuracy : 0.5396
  F1-score : 0.6727

  Confusion Matrix:
                 Predicted
                 rest  task
  Actual rest    42   275
  Actual task    16   299

              precision    recall  f1-score   support

        rest       0.72      0.13      0.22       317
        task       0.52      0.95      0.67       315

    accuracy                           0.54       632
   mac

In [12]:
# Cell 6: Retrain on FULL training set  →  evaluate on held-out test
print("Re-training on full training set …\n")

# ---- FTS ----
fts_full = FTSEncoder(n_sets=N_SETS, mf_type=MF_TYPE,
                      mf_sigma_scale=MF_SIGMA_SCALE, gbell_b=GBELL_B)
F_train_full = fts_full.fit_transform(X_train)
F_test       = fts_full.transform(X_test)
print(f"FTS features : train {F_train_full.shape}   test {F_test.shape}")

zero_pct_train = (F_train_full == 0).sum() / F_train_full.size * 100
zero_pct_test  = (F_test == 0).sum() / F_test.size * 100
print(f"FTS sparsity : train {zero_pct_train:.1f}% zeros, "
      f"test {zero_pct_test:.1f}% zeros")

# ---- 4 × ANFIS ----
anfis_models: list = []
ap_train_full = np.zeros((len(y_train), 4))
ap_test       = np.zeros((len(y_test),  4))

for s in range(4):
    mdl = ANFIS_TSK(
        n_rules=N_RULES, sigma_scale=SIGMA_SCALE,
        ridge_lambda=RIDGE_LAMBDA, n_epochs=N_EPOCHS,
        lr=LR, random_state=42,
    )
    mdl.fit(F_train_full[:, s * 9:(s + 1) * 9], y_train)
    ap_train_full[:, s] = mdl.predict_proba(
        F_train_full[:, s * 9:(s + 1) * 9]
    )
    ap_test[:, s] = mdl.predict_proba(F_test[:, s * 9:(s + 1) * 9])
    anfis_models.append(mdl)
    print(f"  ANFIS [{SIGNAL_NAMES[s]:4s}]  test proba "
          f"[{ap_test[:, s].min():.3f} .. {ap_test[:, s].max():.3f}]")

print(f"ANFIS outputs: train {ap_train_full.shape}  test {ap_test.shape}")

# ---- BN ----
bn_full = BNFusion(n_bins=N_BINS, alpha=ALPHA)
bn_full.fit(ap_train_full, y_train)

ev_test = bn_full.get_evidence_bins(ap_test)
print(f"BN evidence  : test {ev_test.shape}")

# ---- Test predictions ----
test_posteriors = bn_full.predict_proba(ap_test)
test_preds = (test_posteriors >= best_thr).astype(int)

print_metrics(y_test, test_preds,
              title=f"TEST Results  (threshold = {best_thr:.2f})")

# Also show CPT tables learned by BN
print("\nBN CPT tables (full-train fit):")
for sig, tbl in bn_full.get_cpt_table().items():
    print(f"  {sig}:")
    for cls, bins in tbl.items():
        print(f"    P(bin | {cls:4s}) = {bins}")

Re-training on full training set …

FTS features : train (632, 36)   test (193, 36)
FTS sparsity : train 48.3% zeros, test 53.1% zeros
  ANFIS [Temp]  test proba [0.610 .. 0.640]
  ANFIS [HR  ]  test proba [0.517 .. 0.655]
  ANFIS [GSR ]  test proba [0.593 .. 0.643]
  ANFIS [RR  ]  test proba [0.489 .. 0.739]
ANFIS outputs: train (632, 4)  test (193, 4)
BN evidence  : test (193, 4)

  TEST Results  (threshold = 0.23)
  Accuracy : 0.5544
  F1-score : 0.6861

  Confusion Matrix:
                 Predicted
                 rest  task
  Actual rest    13    82
  Actual task     4    94

              precision    recall  f1-score   support

        rest       0.76      0.14      0.23        95
        task       0.53      0.96      0.69        98

    accuracy                           0.55       193
   macro avg       0.65      0.55      0.46       193
weighted avg       0.65      0.55      0.46       193


BN CPT tables (full-train fit):
  Temp:
    P(bin | rest) = {'low': 0.3563, 'med':

In [13]:
# Cell 7: Explainability Trace — one test sample, full pipeline walkthrough
SAMPLE = 0
true_label = "task" if y_test[SAMPLE] == 1 else "rest"
FUZZY = ["Low", "Med", "High"]

print(f"{'=' * 58}")
print(f"  EXPLAINABILITY TRACE  —  Test sample #{SAMPLE}")
print(f"  True label : {true_label}")
print(f"{'=' * 58}")

# --- 1. Transition matrices (one per signal) ---
print("\n  1) FTS Transition Matrices (3×3, normalised):\n")
for s, name in enumerate(SIGNAL_NAMES):
    T = fts_full.get_transition_matrix(X_test[SAMPLE], s)
    print(f"     {name}:")
    print(f"     {'':8s}  {'->Low':>8s} {'->Med':>8s} {'->High':>8s}")
    for i, fl in enumerate(FUZZY):
        print(f"     {fl:8s}  {T[i,0]:8.4f} {T[i,1]:8.4f} {T[i,2]:8.4f}")
    print()

# --- 2. ANFIS p(task) per signal ---
print("  2) ANFIS P(task) per signal:")
for s, name in enumerate(SIGNAL_NAMES):
    print(f"     {name:6s} : {ap_test[SAMPLE, s]:.4f}")

# --- 3. BN evidence bins ---
explanation = bn_full.explain_sample(ap_test[SAMPLE])
print("\n  3) BN Evidence Bins (discretised ANFIS outputs):")
for name, bname in explanation["evidence_bins"].items():
    print(f"     {name:6s} : {bname}")

# --- 4. BN posterior ---
print(f"\n  4) BN Posterior:")
print(f"     P(rest | evidence) = {explanation['posterior_rest']:.4f}")
print(f"     P(task | evidence) = {explanation['posterior_task']:.4f}")
pred = "task" if explanation["posterior_task"] >= best_thr else "rest"
print(f"     Decision (thr={best_thr:.2f}) : {pred}")

# --- 5. Signal contributions ---
print(f"\n  5) Signal Contributions (log-likelihood ratio: task vs rest):")
cpts = bn_full.get_cpt_table()
for s, name in enumerate(SIGNAL_NAMES):
    bname = explanation["evidence_bins"][name]
    ll_task = explanation["log_likelihoods"][1][name]
    ll_rest = explanation["log_likelihoods"][0][name]
    ratio = ll_task - ll_rest
    arrow = "^ pushes TASK" if ratio > 0 else "v pushes REST"
    print(f"     {name:6s} (bin={bname:4s})  "
          f"log-ratio = {ratio:+.4f}   {arrow}")
    print(f"            CPT: P({bname}|task)={cpts[name]['task'][bname]:.4f}"
          f"  P({bname}|rest)={cpts[name]['rest'][bname]:.4f}")

print(f"\n{'=' * 58}")
print(f"  INTERPRETATION:")
print(f"  Signals with positive log-ratio favour 'task';")
print(f"  negative log-ratio favour 'rest'.")
print(f"  The BN multiplies these likelihoods with the prior")
print(f"  to produce the final posterior.")
print(f"{'=' * 58}")

  EXPLAINABILITY TRACE  —  Test sample #0
  True label : task

  1) FTS Transition Matrices (3×3, normalised):

     Temp:
                  ->Low    ->Med   ->High
     Low         0.0000   0.0000   0.0000
     Med         0.0000   0.0000   0.0000
     High        0.0000   0.0000   1.0000

     HR:
                  ->Low    ->Med   ->High
     Low         0.5319   0.1896   0.0000
     Med         0.1909   0.0876   0.0000
     High        0.0000   0.0000   0.0000

     GSR:
                  ->Low    ->Med   ->High
     Low         0.0000   0.0000   0.0000
     Med         0.0000   0.6310   0.1633
     High        0.0000   0.1634   0.0423

     RR:
                  ->Low    ->Med   ->High
     Low         0.0000   0.0000   0.0000
     Med         0.0000   0.2672   0.2255
     High        0.0000   0.2136   0.2937

  2) ANFIS P(task) per signal:
     Temp   : 0.6207
     HR     : 0.6401
     GSR    : 0.6147
     RR     : 0.6725

  3) BN Evidence Bins (discretised ANFIS outputs):
     T

In [14]:
# Cell 8: Run experiments via CLI script (inside notebook)
# -----------------------------------------------------------
# Clear cached project modules so %run picks up the LATEST code
import sys
for _m in ['fts_encoder', 'anfis_tsk', 'bn_fusion', 'eval_utils', 'run_experiments']:
    sys.modules.pop(_m, None)

# === SINGLE mode with preset ===
%run run_experiments.py --mode single --exp A1

# --- Other examples (uncomment to use) ---
%run run_experiments.py --mode single --exp A2
# %run run_experiments.py --mode single --n_sets 4 --mf_type gaussian --lr 0.003 --n_bins 5
# %run run_experiments.py --mode single --exp A1 --primary_metric f1
# %run run_experiments.py --mode grid --primary_metric accuracy

[INFO] Applied preset 'A1': {'n_sets': 3, 'mf_type': 'triangular', 'mf_sigma_scale': 0.5, 'gbell_b': 2.0, 'fts_norm': 'total_sum', 'n_rules': 8, 'sigma_scale': 1.5, 'ridge_lambda': 0.1, 'n_epochs': 15, 'lr': 0.001, 'n_bins': 3, 'alpha': 1.0, 'k_folds': 5}

[Loading data from c:\Users\Lenovo\Desktop\anfis-bn-code\last_30s_segments]
X_train:(632, 4, 30)  X_test:(193, 4, 30)
y_train  rest=317  task=315
y_test   rest=95   task=98

  MODE: SINGLE

[1] Running 5-fold GroupKFold CV ...
[2] Tuning threshold on OOF (metric=accuracy) ...
    Best threshold = 0.465
    OOF accuracy = 0.587
    OOF accuracy = 0.587,  OOF F1 = 0.6087
[3] Retraining on full train -> evaluating test ...
[TEST] acc=0.6736 f1=0.6595

  TEST SET RESULTS
  Accuracy : 0.6736
  F1-score : 0.6595

  Confusion Matrix:
                 Predicted
                 rest  task
  Actual rest    69    26
  Actual task    37    61

              precision    recall  f1-score   support

        rest       0.65      0.73      0.69    

In [15]:
%run run_experiments.py --mode grid --primary_metric accuracy


[Loading data from c:\Users\Lenovo\Desktop\anfis-bn-code\last_30s_segments]
X_train:(632, 4, 30)  X_test:(193, 4, 30)
y_train  rest=317  task=315
y_test   rest=95   task=98

  MODE: GRID SEARCH (limited)
  Total configurations to run: 100
  Per-config TEST eval: ON (diagnostic)

[Grid 1/100] mf=gbell n_sets=4 sigma_mf=0.4 gbell_b=3.0 norm=total_sum rules=12 sig=1.5 ridge=0.01 ep=10 lr=0.005 bins=5 alpha=0.5  | elapsed=0s ETA=0s
         -> OOF acc=0.5459 f1=0.4033 thr=0.595 | TEST acc=0.5285 f1=0.3636
[Grid 2/100] mf=trapezoid n_sets=4 sigma_mf=0.5 gbell_b=2.0 norm=row rules=6 sig=0.8 ridge=0.1 ep=20 lr=0.005 bins=5 alpha=0.5  | elapsed=3s ETA=132s
         -> OOF acc=0.5775 f1=0.5795 thr=0.47 | TEST acc=0.6425 f1=0.6497
[Grid 3/100] mf=gbell n_sets=3 sigma_mf=0.4 gbell_b=3.0 norm=row rules=10 sig=0.8 ridge=0.0 ep=10 lr=0.005 bins=5 alpha=0.5  | elapsed=5s ETA=149s
         -> ERROR: Singular matrix
[Grid 4/100] mf=gbell n_sets=4 sigma_mf=0.6 gbell_b=1.5 norm=row rules=12 sig=1.5 ridg

In [16]:
%run run_experiments.py --mode grid --primary_metric accuracy


[Loading data from c:\Users\Lenovo\Desktop\anfis-bn-code\last_30s_segments]
X_train:(632, 4, 30)  X_test:(193, 4, 30)
y_train  rest=317  task=315
y_test   rest=95   task=98

  MODE: GRID SEARCH (limited)
  Total configurations to run: 100
  Per-config TEST eval: ON (diagnostic)

[Grid 1/100] mf=gbell n_sets=4 sigma_mf=0.4 gbell_b=3.0 norm=total_sum rules=12 sig=1.5 ridge=0.01 ep=10 lr=0.005 bins=5 alpha=0.5  | elapsed=0s ETA=0s
         -> OOF acc=0.5459 f1=0.4033 thr=0.595 | TEST acc=0.5285 f1=0.3636
[Grid 2/100] mf=trapezoid n_sets=4 sigma_mf=0.5 gbell_b=2.0 norm=row rules=6 sig=0.8 ridge=0.1 ep=20 lr=0.005 bins=5 alpha=0.5  | elapsed=3s ETA=131s
         -> OOF acc=0.5775 f1=0.5795 thr=0.47 | TEST acc=0.6425 f1=0.6497
[Grid 3/100] mf=gbell n_sets=3 sigma_mf=0.4 gbell_b=3.0 norm=row rules=10 sig=0.8 ridge=0.0 ep=10 lr=0.005 bins=5 alpha=0.5  | elapsed=5s ETA=153s
         -> ERROR: Singular matrix
[Grid 4/100] mf=gbell n_sets=4 sigma_mf=0.6 gbell_b=1.5 norm=row rules=12 sig=1.5 ridg

In [17]:
%run run_experiments.py --mode grid --primary_metric accuracy --max_configs 300


[Loading data from c:\Users\Lenovo\Desktop\anfis-bn-code\last_30s_segments]
X_train:(632, 4, 30)  X_test:(193, 4, 30)
y_train  rest=317  task=315
y_test   rest=95   task=98

  MODE: GRID SEARCH (limited)
  Total configurations to run: 300
  Per-config TEST eval: ON (diagnostic)

[Grid 1/300] mf=gbell n_sets=4 sigma_mf=0.4 gbell_b=3.0 norm=total_sum rules=12 sig=1.5 ridge=0.01 ep=10 lr=0.005 bins=5 alpha=0.5  | elapsed=0s ETA=0s
         -> OOF acc=0.5459 f1=0.4033 thr=0.595 | TEST acc=0.5285 f1=0.3636
[Grid 2/300] mf=trapezoid n_sets=4 sigma_mf=0.5 gbell_b=2.0 norm=row rules=6 sig=0.8 ridge=0.1 ep=20 lr=0.005 bins=5 alpha=0.5  | elapsed=3s ETA=378s
         -> OOF acc=0.5775 f1=0.5795 thr=0.47 | TEST acc=0.6425 f1=0.6497
[Grid 3/300] mf=gbell n_sets=3 sigma_mf=0.4 gbell_b=3.0 norm=row rules=10 sig=0.8 ridge=0.0 ep=10 lr=0.005 bins=5 alpha=0.5  | elapsed=4s ETA=439s
         -> ERROR: Singular matrix
[Grid 4/300] mf=gbell n_sets=4 sigma_mf=0.6 gbell_b=1.5 norm=row rules=12 sig=1.5 ridg